# 3. Data Preparation

This phase is essentially already included in the modeling pipeline.
The pipeline is intentionally small as we were inspired by the success of the mnist cnn notebook (`cnn-for-mnist-classification.ipynb`) where most "preparation" for image classification problems is
how the data is **loaded** and **augmented** at training time.

---

## 3.1 Manual label cleaning

### What we did

Before the final training run, the labels were cleaned manually using the **predictions of an
earlier CNN training run** as a guide. We ran it 3-4 times Concretely:

1. Train an initial CNN on the raw scraped data: `data/skins/good/` and
   `data/skins/bad/spiderman/`.
2. Predict on the full dataset.
3. Inspect the strongest **false positives** (predicted Spider-Man with high confidence,
   labeled normal) and **false negatives** (predicted normal with high confidence, labeled
   Spider-Man). These were the cases where the model and the label disagreed most strongly.
4. **Manually re-classify these by eye**, moving images between folders:
   - Real Spider-Man skins that had slipped into the Kaggle "normal" sample → moved to
     `bad/spiderman_cleaned/`.
   - Normal-looking skins that the keyword scraper had mistakenly picked up (the search engine
     on `minecraftskins.com` is not perfect) → moved to `good_cleaned/`.
5. Retrain on the cleaned folders. This is what the Modeling and Evaluation notebooks use.

### Why we did it

The first CNN's biggest mistakes were not random — they were **systematic**, and they pointed
directly at mislabeled training examples. Cleaning these flips the worst training labels and
removes the corresponding contradictory gradient signal, which lets the model converge to a
more confident decision boundary.

### What this is

This is a **manual iteration of active learning**, with the model itself as the query
strategy (high confidence but disagreement). It worked well here because
the dataset is small enough that one human reviewer can inspect a hundred contested cases
in a short time.

For a production system this would need to be automated:
- Schedule periodic retraining whenever new banned-player data becomes available.
- Surface the most uncertain or most disagreed-with predictions in a moderator tool.
- Let moderators relabel through that tool; feed the corrections back into the training set.
- Optionally combine with perceptual-hash deduplication to avoid the same skin being relabeled
  multiple times.

---

Data preparation is small by design, mos steps were included in the modeling notebook:

1. **Cleaning:** one manual CNN-in-the-loop relabeling pass (active learning by hand).
2. **Loading:** 64×64 RGBA, normalized to `[0, 1]`.
3. **Splitting:** stratified 70 / 15 / 15 with a fixed seed.
4. **Weighting:** balanced class weights applied as per-sample weights at loss time.
5. **Augmentation:** color shift + Gaussian noise *only*, applied inside the model graph.

Everything else stays raw, because the value of this dataset is in its full **structure**
and we do not want to wash that away.